# AdaBoost Hyperparameters ,Hyperparameter Tuning & Bagging VS Boosting

## 1. Core Hyperparameters

### A. `estimator` (Previously `base_estimator`)

**Definition:**  
Specifies the weak learner used to build the boosting sequence.

**Default:**  
If left unspecified, AdaBoost uses:

```python
DecisionTreeClassifier(max_depth=1)
```

This is known as a **Decision Stump**.

**Alternative Options:**

- `DecisionTreeClassifier`
- `LogisticRegression`
- `SVC`

**Note:**  
Algorithms like `KNeighborsClassifier` cannot be used because they do not support sample weighting.

**Practical Recommendation:**  
In most real-world applications, Decision Stumps remain the preferred choice because they maximize diversity between boosting rounds.

---

### B. `n_estimators`

**Definition:**  
The maximum number of weak learners (stumps) trained sequentially.

**Behavior:**

| Value | Effect |
|---------|---------|
| Very Low (e.g., 1) | Underfitting |
| Moderate | Good generalization |
| Very High (e.g., 1000+) | Risk of overfitting |

**Important:**  
AdaBoost can stop early if a learner achieves perfect classification on the weighted dataset.

---

### C. `learning_rate`

**Definition:**  
Controls how much influence each weak learner contributes to the final ensemble.

**Default:** `1.0`

Lower values slow down learning and act as a regularizer.

---

## 2. Learning Rate & Shrinkage

The learning rate modifies the influence of each weak learner:

$$
\alpha_{m,\text{modified}}
=
\text{learning\_rate}
\times
\left[
\frac{1}{2}
\ln
\left(
\frac{1-\epsilon_m}{\epsilon_m}
\right)
\right]
$$

Where:

- $\alpha_m$ = weight assigned to the weak learner
- $\epsilon_m$ = weighted classification error

### Effect of Lower Learning Rate

When learning rate decreases:

- Misclassified samples receive smaller weight increases
- Correctly classified samples receive smaller weight decreases
- Learning becomes slower and more stable
- Overfitting risk decreases

### Practical Trade-Off

| Learning Rate | Typical Effect |
|---------------|---------------|
| High | Faster learning, higher overfitting risk |
| Low | Slower learning, better generalization |

A common strategy is:

- Lower `learning_rate`
- Increase `n_estimators`

This often improves performance.

---

## 3. AdaBoost Algorithms

### `SAMME`

Uses hard class predictions.

Example:

- Class A
- Class B
- Class C

Only the predicted class label is used.

---

### `SAMME.R`

Uses class probabilities (`predict_proba()`).

Example:

| Class | Probability |
|---------|---------|
| A | 0.80 |
| B | 0.15 |
| C | 0.05 |

Because it utilizes probability information, it generally:

- Converges faster
- Requires fewer estimators
- Achieves lower test error

---

## 4. Hyperparameter Tuning Using GridSearchCV

The most commonly tuned parameters are:

- `n_estimators`
- `learning_rate`
- `algorithm`

Example search space:

```python
param_grid = {
    'n_estimators': [10, 50, 100, 500],
    'learning_rate': [0.0001, 0.001, 0.01, 0.1, 1.0],
    'algorithm': ['SAMME', 'SAMME.R']
}
```

### Why Grid Search?

Grid Search systematically tests combinations of parameters and finds the configuration producing the best cross-validation score.

### Benefits

- Removes guesswork
- Improves model performance
- Finds the best balance between underfitting and overfitting

---

# Summary Table

| Hyperparameter | Meaning |
|---------------|----------|
| `estimator` | Weak learner used for boosting |
| `n_estimators` | Number of sequential weak learners |
| `learning_rate` | Controls contribution of each learner |
| `algorithm` | Determines how learner weights are computed (`SAMME` or `SAMME.R`) |
| `cv` | Number of folds used during cross-validation |
| `n_jobs` | Number of CPU cores used for parallel processing |
| `scoring` | Metric used to evaluate parameter combinations |
| `param_grid` | Search space of hyperparameter values |

# Bagging vs. Boosting

Ensemble Learning combines multiple models to create a stronger predictive model. Both Bagging and Boosting improve model performance, but they solve different problems and use different training strategies.

---

# 1. Key Differences Between Bagging and Boosting

## Difference 1: Type of Base Learners Used

### Bagging

- Designed to reduce **Variance (Overfitting)**.
- Uses **strong learners** as base models.
- Typically uses deep, fully-grown Decision Trees.

**Reason:**
A single deep tree has:

- Low Bias
- High Variance

Bagging averages many such trees to reduce variance while maintaining low bias.

---

### Boosting

- Designed to reduce **Bias (Underfitting)**.
- Uses **weak learners** as base models.
- Typically uses shallow trees or Decision Stumps (`max_depth = 1`).

**Reason:**
A single stump has:

- High Bias
- Low Variance

Boosting combines many weak learners sequentially to gradually reduce bias.

---

## Difference 2: Training Architecture

### Bagging → Parallel Training

- Models are trained independently.
- Each model receives a bootstrap sample of the dataset.
- No model depends on another model.

#### Workflow

Dataset → Tree 1

Dataset → Tree 2

Dataset → Tree 3

Dataset → Tree 4

(All trees train simultaneously)

---

### Boosting → Sequential Training

- Models are trained one after another.
- Each new model learns from mistakes made by previous models.
- Later models focus more on difficult observations.

#### Workflow

Model 1 → Errors

↓

Model 2 → Errors

↓

Model 3 → Errors

↓

Model 4

Each model depends on the previous model.

---

## Difference 3: Final Prediction Mechanism

### Bagging

Uses **Equal Voting**.

Each model has the same importance.

#### Classification

Uses Majority Voting.

Example:

Tree 1 → Class A

Tree 2 → Class A

Tree 3 → Class B

Prediction = Class A

---

#### Regression

Uses Simple Averaging.

$$
Prediction
=
\frac{
Pred_1 + Pred_2 + Pred_3 + \cdots + Pred_n
}{n}
$$

---

### Boosting

Uses **Weighted Voting**.

Each model receives a weight based on its performance.

Better-performing models receive higher influence.

#### Classification

$$
Final\ Prediction
=
sign\left(
\sum_{m=1}^{M}
\alpha_m h_m(x)
\right)
$$

Where:

- $h_m(x)$ = prediction from model $m$
- $\alpha_m$ = importance of model $m$

Models with higher $\alpha$ contribute more to the final prediction.

---

# Summary Table

| Feature | Bagging | Boosting |
|----------|----------|----------|
| Primary Goal | Reduce Variance | Reduce Bias |
| Main Problem Solved | Overfitting | Underfitting |
| Base Learners | Strong Learners | Weak Learners |
| Typical Tree Depth | Deep Trees | Decision Stumps |
| Training Method | Parallel | Sequential |
| Model Dependency | Independent | Dependent |
| Data Strategy | Bootstrap Sampling | Re-weight Errors / Residual Learning |
| Final Aggregation | Equal Voting | Weighted Voting |
| Classification Output | Majority Vote | Weighted Vote |
| Regression Output | Average Prediction | Weighted Sum |
| Overfitting Risk | Lower | Higher |
| Training Speed | Faster (Parallelizable) | Slower (Sequential) |
| Examples | Random Forest, Bagging Classifier | AdaBoost, Gradient Boosting, XGBoost |

---

# One-Line Memory Trick

| Bagging | Boosting |
|----------|----------|
| Many Strong Models → Reduce Variance | Many Weak Models → Reduce Bias |
| Parallel Learning | Sequential Learning |
| Equal Voting | Weighted Voting |
| Random Forest | AdaBoost / Gradient Boosting |m